In [ ]:
import os
os.kill(os.getpid(), 9)


In [ ]:
!pip install scikit-surprise

In [ ]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import cross_validate

# Load MovieLens dataset
data = Dataset.load_builtin('ml-100k')

# Build and evaluate an SVD-based recommender
algo = SVD()
cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)


Dataset ml-100k could not be found. Do you want to download it? [Y/n] y
Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.9369  0.9247  0.9461  0.9382  0.9404  0.9373  0.0070  
MAE (testset)     0.7371  0.7319  0.7450  0.7392  0.7389  0.7384  0.0042  
Fit time          1.34    1.62    1.91    1.46    1.27    1.52    0.23    
Test time         0.12    0.30    0.19    0.11    0.20    0.18    0.07    


{'test_rmse': array([0.93688832, 0.92473638, 0.94612864, 0.93818967, 0.94041464]),
 'test_mae': array([0.73708391, 0.73188493, 0.74503973, 0.73922139, 0.73885252]),
 'fit_time': (1.337648630142212,
  1.6192872524261475,
  1.905454397201538,
  1.4583580493927002,
  1.2735176086425781),
 'test_time': (0.12450861930847168,
  0.2992234230041504,
  0.18596124649047852,
  0.1064596176147461,
  0.19899702072143555)}

In [ ]:
from surprise import Dataset
from surprise.model_selection import train_test_split

# Reload the dataset and build trainset
data = Dataset.load_builtin('ml-100k')
trainset = data.build_full_trainset()
algo.fit(trainset)

# Get list of all item ids (movies)
all_items = trainset.all_items()
all_item_inner_ids = [iid for iid in all_items]
all_item_raw_ids = [trainset.to_raw_iid(inner_id) for inner_id in all_item_inner_ids]

# Get list of items user has already rated
user_id = str(196)
user_rated_items = [j for (j, _) in trainset.ur[trainset.to_inner_uid(user_id)]]

# Get items user has NOT rated
items_to_predict = [iid for iid in all_item_inner_ids if iid not in user_rated_items]

# Predict ratings for all unseen items
predictions = [algo.predict(uid=user_id, iid=trainset.to_raw_iid(iid)) for iid in items_to_predict]

# Sort by estimated rating
top_n = sorted(predictions, key=lambda x: x.est, reverse=True)[:5]

# Show top 5 recommendations
print(f"\nTop 5 movie recommendations for user {user_id}:")
for pred in top_n:
    print(f"Movie ID: {pred.iid}, Predicted Rating: {pred.est:.2f}")
